In [ ]:
import matplotlib
matplotlib.use("Agg")
import os
import copy
import gc
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Using Device = {str(device).title()}")

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

classnames = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
num_classes = len(classnames)

def preprocess_train(image, label):
    image = tf.image.resize(image, (128, 128))
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = (image - 0.5) / 0.5  
    return image, label

def preprocess_test(image, label):
    image = tf.image.resize(image, (128, 128))
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - 0.5) / 0.5
    return image, label

batch_size = 50

train_ds = tf.data.Dataset.from_tensor_slices((x_train_raw[:1000], y_train_raw[:1000]))
train_ds = train_ds.shuffle(1000).map(preprocess_train).batch(batch_size).drop_remainder(True).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test_raw[:200], y_test_raw[:200]))
test_ds = test_ds.map(preprocess_test).batch(batch_size).prefetch(tf.data.AUTOTUNE)

class ComputerVision(keras.Model):
    def __init__(self, num_classes=num_classes):
        super().__init__()
        
        self.features = keras.Sequential([
            keras.layers.Input(shape=(128, 128, 3)),
            
            keras.layers.ZeroPadding2D(padding=((2, 2), (2, 2))),
            keras.layers.Conv2D(filters=32, kernel_size=4, strides=3),
            keras.layers.BatchNormalization(),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.MaxPooling2D(pool_size=2, strides=2),
            
            keras.layers.ZeroPadding2D(padding=((6, 6), (6, 6))),
            keras.layers.Conv2D(filters=87, kernel_size=5, strides=3),
            keras.layers.BatchNormalization(),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.MaxPooling2D(pool_size=2, strides=3),
            
            # Convolutional Stage 3
            keras.layers.ZeroPadding2D(padding=((10, 10), (10, 10))),
            keras.layers.Conv2D(filters=126, kernel_size=9, strides=4),
            keras.layers.BatchNormalization(),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.MaxPooling2D(pool_size=3, strides=5),
            
            keras.layers.ZeroPadding2D(padding=((1, 2), (1, 2))),
            keras.layers.Conv2D(filters=256, kernel_size=2, strides=3),
            keras.layers.BatchNormalization(),
            keras.layers.LeakyReLU(negative_slope=0.01),
            
            keras.layers.GlobalAveragePooling2D()
        ])
        
        self.classifier = keras.Sequential([
            keras.layers.Dense(units=64),
            keras.layers.LeakyReLU(negative_slope=0.45),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(rate=0.4),
            keras.layers.Dense(units=num_classes)
        ])
        
    def call(self, x, training=False):
        x = self.features(x, training=training)
        return self.classifier(x, training=training)


losses = []
accuracies = []

def grid_search(epoch_count_list, learning_rates, seed_val):
    best_loss = float("inf")
    best_state = None
    
    for lr in learning_rates:
        for count in epoch_count_list:
            print(f"\nEvaluating Configuration : Lr = {lr} | Epochs = {count}")
            tf.keras.utils.set_random_seed(int(seed_val))
            
            model = ComputerVision()
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=[keras.metrics.SparseCategoricalAccuracy()]
            )
            
            for e in tqdm(range(count), desc="Training Loops"):
                model.fit(train_ds, verbose=0)
                
            test_loss, test_acc = model.evaluate(test_ds, verbose=0)
            losses.append(test_loss)
            accuracies.append(test_acc)
            
            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.get_weights())
                
            del model
            keras.backend.clear_session()
            gc.collect()
            
    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies) * 100:.2f} %")
    return best_state

epoch_counts_list = [100,50]
rates_list = [0.1,0.01]
seed = 42

best_state = grid_search(epoch_count_list=epoch_counts_list, learning_rates=rates_list, seed_val=seed)

fresh_model = ComputerVision()
fresh_model.build((None, 128, 128, 3))
fresh_model.set_weights(best_state)

while True:
    user_image_path = input("-- Enter the Absolute Path to an Image File ( or type 'exit' to quit ) : ").strip()
    if user_image_path.lower() == "exit" or not user_image_path:
        print("Exiting Interactive Loop ...")
        break
        
    if not os.path.exists(user_image_path):
        print(f"File Path does not Exist : {user_image_path}\n")
        continue
        
    try:
       
        display_img = keras.utils.load_img(user_image_path)
        img_array = keras.utils.img_to_array(display_img)
        
        inference_tensor = tf.image.resize(img_array, (128, 128))
        inference_tensor = tf.cast(inference_tensor, tf.float32) / 255.0
        inference_tensor = (inference_tensor - 0.5) / 0.5
        
        expanded_tensor = tf.expand_dims(inference_tensor, axis=0)
        
        logits = fresh_model(expanded_tensor, training=False)
        probabilities = tf.nn.softmax(logits, axis=1).numpy()
        pred_idx = int(np.argmax(probabilities))
        confidence = probabilities[pred_idx] * 100
        
        print(f"\n● Predicted Class = {classnames[pred_idx].title()}")
        print(f"● Confidence Score = {confidence:.2f}%\n")
        
        plt.figure(figsize=(5, 4))
        plt.imshow(img_array.astype(np.uint8))  # Display the original un-normalized pixel layout arrays
        plt.title(f"Prediction : {classnames[pred_idx]} | Confidence : {confidence:.2f}%")
        plt.axis("off")
        
    except Exception as e:
        print(f"❌ Failed to process external image file | Error : {e}\n")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import os
import copy
import gc
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Using Device = {str(device).title()}")

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

classnames = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
num_classes = len(classnames)

def preprocess_train(image, label):
    image = tf.image.resize(image, (128, 128))
    image = keras.applications.mobilenet_v2.preprocess_input(tf.cast(image, tf.float32))
    image = tf.image.random_flip_left_right(image)
    return image, label

def preprocess_test(image, label):
    image = tf.image.resize(image, (128, 128))
    image = keras.applications.mobilenet_v2.preprocess_input(tf.cast(image, tf.float32))
    return image, label

batch_size = 50

train_ds = tf.data.Dataset.from_tensor_slices((x_train_raw[:1000], y_train_raw[:1000]))
train_ds = train_ds.shuffle(1000).map(preprocess_train).batch(batch_size).drop_remainder(True).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test_raw[:200], y_test_raw[:200]))
test_ds = test_ds.map(preprocess_test).batch(batch_size).prefetch(tf.data.AUTOTUNE)

class PretrainedComputerVision(keras.Model):
    def __init__(self, num_classes=num_classes):
        super().__init__()
        
        self.base_model = keras.applications.MobileNetV2(
            input_shape=(128, 128, 3),
            include_top=False,  #
            weights='imagenet'
        )
        
        self.base_model.trainable = False
        
        self.global_pool = keras.layers.GlobalAveragePooling2D()
        
        self.classifier = keras.Sequential([
            keras.layers.Dense(units=64),
            keras.layers.LeakyReLU(negative_slope=0.45),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(rate=0.4),
            keras.layers.Dense(units=num_classes)
        ])
        
    def call(self, x, training=False):
        x = self.base_model(x, training=False)
        x = self.global_pool(x)
        return self.classifier(x, training=training)

losses = []
accuracies = []

def grid_search(epoch_count_list, learning_rates, seed_val):
    best_loss = float("inf")
    best_state = None
    
    for lr in learning_rates:
        for count in epoch_count_list:
            print(f"\nEvaluating Configuration : Lr = {lr} | Epochs = {count}")
            tf.keras.utils.set_random_seed(int(seed_val))
            
            model = PretrainedComputerVision()
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=[keras.metrics.SparseCategoricalAccuracy()]
            )
            
            for e in tqdm(range(count), desc="Training Loops"):
                model.fit(train_ds, verbose=0)
                
            test_loss, test_acc = model.evaluate(test_ds, verbose=0)
            losses.append(test_loss)
            accuracies.append(test_acc)
            
            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.get_weights())
                
            del model
            keras.backend.clear_session()
            gc.collect()
            
    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies) * 100:.2f} %")
    return best_state

epoch_counts_list = [100,50]
rates_list = [0.1,0.01]
seed = 42

best_state = grid_search(epoch_count_list=epoch_counts_list, learning_rates=rates_list, seed_val=seed)

fresh_model = PretrainedComputerVision()
fresh_model.build((None, 128, 128, 3))
fresh_model.set_weights(best_state)

while True:
    user_image_path = input("-- Enter the Absolute Path to an Image File ( or type 'exit' to quit ) : ").strip()
    if user_image_path.lower() == "exit" or not user_image_path:
        print("Exiting Interactive Loop ...")
        break
        
    if not os.path.exists(user_image_path):
        print(f"⚠️ File Path does not Exist: {user_image_path}\n")
        continue
        
    try:
        display_img = keras.utils.load_img(user_image_path)
        img_array = keras.utils.img_to_array(display_img)
        
        inference_tensor = tf.image.resize(img_array, (128, 128))
        inference_tensor = keras.applications.mobilenet_v2.preprocess_input(inference_tensor)
        
        expanded_tensor = tf.expand_dims(inference_tensor, axis=0)
        
        logits = fresh_model(expanded_tensor, training=False)
        probabilities = tf.nn.softmax(logits, axis=1).numpy()
        pred_idx = int(np.argmax(probabilities))
        confidence = probabilities[pred_idx] * 100
        
        print(f"\n● Predicted Class = {classnames[pred_idx].title()}")
        print(f"● Confidence Score = {confidence:.2f}%\n")
        
        plt.figure(figsize=(5, 4))
        plt.imshow(img_array.astype(np.uint8))  
        plt.title(f"Prediction : {classnames[pred_idx]} | Confidence : {confidence:.2f}%")
        plt.axis("off")
        
    except Exception as e:
        print(f"❌ Failed to process external image file | Error : {e}\n")